# Boticário & Natura Catalog Data Analysis

Comprehensive analysis of cosmetics retail data extracted from O Boticário and Natura sales catalogs.

**Dataset:** 40,000+ products from 33 catalog PDFs  
**Brands:** O Boticário, Natura  
**Extraction Method:** Google Gemini Vision AI

---

## 1. Setup & Configuration

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML & NLP
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

# Additional
from wordcloud import WordCloud
import networkx as nx

print("Libraries loaded successfully!")

In [ ]:
# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100

# Color palettes
BOTICARIO_COLORS = ['#006847', '#00A859', '#7DC242']
NATURA_COLORS = ['#F57C00', '#FF9800', '#FFB74D']
BRAND_COLORS = {'boticario': '#006847', 'natura': '#F57C00'}

# Seaborn settings
sns.set_palette('Set2')

print("Visualization settings configured!")

### Helper Functions

In [ ]:
def parse_volume(volume_str):
    """
    Extract numeric volume/weight from string.
    Examples: '200ml' -> 200, '400g' -> 400, '2x100ml' -> 200
    """
    if pd.isna(volume_str) or volume_str == '':
        return None, None
    
    volume_str = str(volume_str).lower().strip()
    
    # Handle multipliers (e.g., "2x100ml")
    multiplier_match = re.search(r'(\d+)\s*x\s*(\d+\.?\d*)', volume_str)
    if multiplier_match:
        multiplier = float(multiplier_match.group(1))
        value = float(multiplier_match.group(2))
        total = multiplier * value
    else:
        # Extract numeric value
        num_match = re.search(r'(\d+\.?\d*)', volume_str)
        total = float(num_match.group(1)) if num_match else None
    
    # Determine unit
    if 'ml' in volume_str or 'l' in volume_str:
        unit = 'ml'
        if 'l' in volume_str and 'ml' not in volume_str:
            total = total * 1000 if total else None  # Convert L to ml
    elif 'g' in volume_str or 'kg' in volume_str:
        unit = 'g'
        if 'kg' in volume_str:
            total = total * 1000 if total else None  # Convert kg to g
    else:
        unit = 'un'
    
    return total, unit


def extract_cycle_number(filename):
    """
    Extract cycle number from filename.
    Example: 'boticario_ciclo_05_...' -> 5
    """
    match = re.search(r'ciclo_(\d+)', str(filename).lower())
    return int(match.group(1)) if match else None


def format_currency(value):
    """Format value as Brazilian Real."""
    return f'R$ {value:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def safe_percentage(numerator, denominator):
    """Calculate percentage safely."""
    return (numerator / denominator * 100) if denominator > 0 else 0


print("Helper functions defined!")

<![CDATA[---
## 2. Data Loading

### Option A: Kaggle Environment (Recommended)
If running on Kaggle, the dataset is automatically available in `/kaggle/input/`.

### Option B: Local Environment with Kaggle API
Use the code below to download the dataset locally using the Kaggle library.]]>

In [ ]:
<![CDATA[# Kaggle Dataset Configuration using kagglehub
import os
import kagglehub

# Dataset identifiers on Kaggle
KAGGLE_DATASETS = {
    'boticario': 'arturovainecwb/o-boticrio-products-catalogs-20252026',
    'natura': 'arturovainecwb/natura-and-co-products-catalogs-20252026'
}

def setup_kaggle_data():
    """
    Setup data directories based on environment (Kaggle or local).
    Returns a list of paths to the data directories.
    """
    data_paths = []
    
    # Check if running on Kaggle
    if os.path.exists('/kaggle/input'):
        print("Running on Kaggle environment")
        kaggle_input = Path('/kaggle/input')
        datasets = list(kaggle_input.iterdir())
        if datasets:
            for ds in datasets:
                print(f"Dataset found: {ds}")
                data_paths.append(ds)
            return data_paths
        else:
            raise FileNotFoundError("No datasets found in /kaggle/input/")
    
    # Local environment - use kagglehub to download
    print("Running locally - downloading datasets via kagglehub...\n")
    
    for brand, dataset_slug in KAGGLE_DATASETS.items():
        try:
            print(f"Downloading {brand.title()} dataset...")
            path = kagglehub.dataset_download(dataset_slug)
            print(f"  -> {path}\n")
            data_paths.append(Path(path))
        except Exception as e:
            print(f"  Error downloading {brand}: {e}\n")
    
    # Fallback if no datasets downloaded
    if not data_paths:
        print("Falling back to local data directory...")
        data_paths.append(Path('../data/output'))
    
    return data_paths

# Setup data directories
DATA_DIRS = setup_kaggle_data()
print(f"\nData directories: {len(DATA_DIRS)} found")
for d in DATA_DIRS:
    print(f"  - {d} (exists: {d.exists()})")]]>

In [ ]:
<![CDATA[def load_catalog_data(data_dirs):
    """
    Load all catalog JSON files from multiple directories and combine into a single DataFrame.
    
    Args:
        data_dirs: List of paths to data directories
    """
    all_products = []
    catalog_metadata = []
    
    # Handle single directory or list of directories
    if not isinstance(data_dirs, list):
        data_dirs = [data_dirs]
    
    for data_dir in data_dirs:
        json_files = list(Path(data_dir).glob('*.json'))
        print(f"Found {len(json_files)} JSON files in {data_dir.name}")
        
        for json_file in json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                catalog = json.load(f)
            
            # Extract brand from filename
            filename = json_file.name.lower()
            brand = 'boticario' if 'boticario' in filename else 'natura'
            cycle = extract_cycle_number(filename)
            
            # Store catalog metadata
            meta = catalog.get('metadata', {})
            meta['source_file'] = json_file.name
            meta['brand'] = brand
            meta['cycle'] = cycle
            catalog_metadata.append(meta)
            
            # Process products
            products = catalog.get('products', [])
            for product in products:
                product['source_file'] = json_file.name
                product['brand'] = brand
                product['cycle'] = cycle
                
                # Flatten promotional rule
                promo_rule = product.get('regra_promocional', {}) or {}
                product['promo_type'] = promo_rule.get('type', 'none')
                product['promo_description'] = promo_rule.get('description', '')
                product['discount_tiers'] = promo_rule.get('discount_tiers', {})
                product['combo_codes'] = promo_rule.get('combo_codes', [])
                
                # Flatten alerts
                alerts = product.get('alertas', []) or []
                product['alert_count'] = len(alerts)
                product['has_errors'] = any(a.get('level') == 'error' for a in alerts)
                product['has_warnings'] = any(a.get('level') == 'warning' for a in alerts)
                
                all_products.append(product)
    
    df = pd.DataFrame(all_products)
    df_meta = pd.DataFrame(catalog_metadata)
    
    return df, df_meta


# Load data from all directories
df, df_meta = load_catalog_data(DATA_DIRS)
print(f"\nLoaded {len(df):,} products from {len(df_meta)} catalogs")]]>

In [ ]:
# Initial data inspection
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Sample data
df.head()

---
## 3. Data Preprocessing

In [ ]:
# Parse volume/weight
df[['volume_numeric', 'volume_unit']] = df['volume_peso'].apply(
    lambda x: pd.Series(parse_volume(x))
)

print("Volume parsing sample:")
df[['volume_peso', 'volume_numeric', 'volume_unit']].dropna().head(10)

In [ ]:
# Data type conversions
numeric_cols = ['preco_regular', 'preco_promocional', 'economia', 'desconto_percentual', 'pagina']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Boolean conversion
if 'promocao_ativa' in df.columns:
    df['promocao_ativa'] = df['promocao_ativa'].fillna(False).astype(bool)

print("Numeric columns converted:")
df[numeric_cols].dtypes

In [ ]:
# Feature engineering

# Price per ml/g (when applicable)
df['price_per_unit'] = np.where(
    df['volume_numeric'] > 0,
    df['preco_regular'] / df['volume_numeric'],
    np.nan
)

# Calculate discount if missing
df['desconto_calculado'] = np.where(
    (df['preco_regular'] > 0) & (df['preco_promocional'].notna()),
    ((df['preco_regular'] - df['preco_promocional']) / df['preco_regular'] * 100),
    df['desconto_percentual']
)

# Price tier based on regular price
df['price_tier'] = pd.cut(
    df['preco_regular'],
    bins=[0, 30, 60, 100, 200, float('inf')],
    labels=['Budget', 'Economy', 'Standard', 'Premium', 'Luxury']
)

# Features count
df['features_count'] = df['caracteristicas'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print("New features created:")
print(df[['price_per_unit', 'desconto_calculado', 'price_tier', 'features_count']].head())

In [ ]:
# Handle missing values summary
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print("Missing Values Summary (top 15):")
missing_summary.head(15)

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

stats = {
    'Total Products': len(df),
    'Unique Products (by code)': df['codigo'].nunique(),
    'Brands': df['brand'].nunique(),
    'Catalogs': df['source_file'].nunique(),
    'Cycles': df['cycle'].nunique(),
    'Categories': df['categoria_normalizada'].nunique(),
    'Product Lines': df['linha'].nunique(),
    'Products with Promotion': df['promocao_ativa'].sum(),
    'Promotion Rate (%)': round(df['promocao_ativa'].mean() * 100, 2),
    'Avg Regular Price': format_currency(df['preco_regular'].mean()),
    'Avg Discount (%)': round(df['desconto_calculado'].mean(), 2),
}

for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
# Brand distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Products per brand
brand_counts = df['brand'].value_counts()
colors = [BRAND_COLORS.get(b, '#333') for b in brand_counts.index]

axes[0].pie(brand_counts, labels=brand_counts.index, autopct='%1.1f%%', 
            colors=colors, explode=[0.02, 0.02], startangle=90)
axes[0].set_title('Products by Brand', fontsize=14, fontweight='bold')

# Catalogs per brand
catalog_counts = df.groupby('brand')['source_file'].nunique()
axes[1].bar(catalog_counts.index, catalog_counts.values, color=colors)
axes[1].set_title('Number of Catalogs by Brand', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Catalogs')

for i, v in enumerate(catalog_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Products per cycle
cycle_products = df.groupby(['brand', 'cycle']).size().reset_index(name='count')

fig = px.bar(
    cycle_products, 
    x='cycle', 
    y='count', 
    color='brand',
    barmode='group',
    title='Products per Cycle by Brand',
    color_discrete_map=BRAND_COLORS,
    labels={'cycle': 'Cycle', 'count': 'Number of Products', 'brand': 'Brand'}
)
fig.update_layout(xaxis_tickmode='linear')
fig.show()

In [ ]:
# Numerical variables distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Regular Price Distribution
df['preco_regular'].hist(bins=50, ax=axes[0, 0], color='steelblue', edgecolor='white')
axes[0, 0].set_title('Regular Price Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Price (R$)')
axes[0, 0].axvline(df['preco_regular'].median(), color='red', linestyle='--', label=f"Median: {format_currency(df['preco_regular'].median())}")
axes[0, 0].legend()

# Discount Distribution
df[df['desconto_calculado'] > 0]['desconto_calculado'].hist(bins=30, ax=axes[0, 1], color='coral', edgecolor='white')
axes[0, 1].set_title('Discount Distribution (when > 0%)', fontweight='bold')
axes[0, 1].set_xlabel('Discount (%)')
axes[0, 1].axvline(df[df['desconto_calculado'] > 0]['desconto_calculado'].median(), color='red', linestyle='--', label=f"Median: {df[df['desconto_calculado'] > 0]['desconto_calculado'].median():.1f}%")
axes[0, 1].legend()

# Volume Distribution
df[df['volume_numeric'].notna() & (df['volume_numeric'] < 1000)]['volume_numeric'].hist(bins=40, ax=axes[1, 0], color='seagreen', edgecolor='white')
axes[1, 0].set_title('Volume Distribution (< 1000ml/g)', fontweight='bold')
axes[1, 0].set_xlabel('Volume (ml/g)')

# Savings Distribution
df[df['economia'] > 0]['economia'].hist(bins=40, ax=axes[1, 1], color='purple', edgecolor='white')
axes[1, 1].set_title('Savings Distribution (when > 0)', fontweight='bold')
axes[1, 1].set_xlabel('Savings (R$)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix for numerical variables
corr_cols = ['preco_regular', 'preco_promocional', 'economia', 'desconto_calculado', 
             'volume_numeric', 'price_per_unit', 'pagina', 'features_count']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', linewidths=0.5, square=True)
plt.title('Correlation Matrix - Numerical Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Pricing & Promotions Analysis

### 5.1 Discount Distribution Analysis

In [ ]:
# Discount statistics by brand
discount_stats = df[df['desconto_calculado'] > 0].groupby('brand')['desconto_calculado'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(2)
discount_stats.columns = ['Products with Discount', 'Mean %', 'Median %', 'Std %', 'Min %', 'Max %']
print("Discount Statistics by Brand:")
discount_stats

In [ ]:
# Discount distribution by brand
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
df_with_discount = df[df['desconto_calculado'] > 0]
sns.boxplot(data=df_with_discount, x='brand', y='desconto_calculado', 
            palette=BRAND_COLORS, ax=axes[0])
axes[0].set_title('Discount Distribution by Brand', fontweight='bold')
axes[0].set_ylabel('Discount (%)')
axes[0].set_xlabel('Brand')

# Violin plot
sns.violinplot(data=df_with_discount, x='brand', y='desconto_calculado', 
               palette=BRAND_COLORS, ax=axes[1])
axes[1].set_title('Discount Distribution (Violin)', fontweight='bold')
axes[1].set_ylabel('Discount (%)')
axes[1].set_xlabel('Brand')

plt.tight_layout()
plt.show()

In [ ]:
# Discount ranges distribution
df['discount_range'] = pd.cut(
    df['desconto_calculado'],
    bins=[-0.1, 0, 10, 20, 30, 40, 50, 100],
    labels=['No Discount', '1-10%', '11-20%', '21-30%', '31-40%', '41-50%', '50%+']
)

discount_range_counts = df.groupby(['brand', 'discount_range']).size().unstack(fill_value=0)

discount_range_counts.T.plot(kind='bar', figsize=(12, 6), color=[BRAND_COLORS['boticario'], BRAND_COLORS['natura']])
plt.title('Discount Range Distribution by Brand', fontsize=14, fontweight='bold')
plt.xlabel('Discount Range')
plt.ylabel('Number of Products')
plt.xticks(rotation=45)
plt.legend(title='Brand')
plt.tight_layout()
plt.show()

In [ ]:
# Discount by category
category_discounts = df[df['desconto_calculado'] > 0].groupby('categoria_normalizada')['desconto_calculado'].agg(
    ['mean', 'median', 'count']
).sort_values('mean', ascending=False)

fig = px.bar(
    category_discounts.reset_index(),
    x='categoria_normalizada',
    y='mean',
    color='count',
    title='Average Discount by Category',
    labels={'categoria_normalizada': 'Category', 'mean': 'Avg Discount (%)', 'count': 'Product Count'},
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 5.2 Price Analysis

In [ ]:
# Price statistics by brand
price_stats = df.groupby('brand')['preco_regular'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(2)
price_stats.columns = ['Products', 'Mean (R$)', 'Median (R$)', 'Std (R$)', 'Min (R$)', 'Max (R$)']
print("Price Statistics by Brand:")
price_stats

In [ ]:
# Price distribution by brand
fig = px.histogram(
    df[df['preco_regular'] < 500],  # Filter outliers for better visualization
    x='preco_regular',
    color='brand',
    nbins=50,
    barmode='overlay',
    opacity=0.7,
    title='Price Distribution by Brand (< R$500)',
    color_discrete_map=BRAND_COLORS,
    labels={'preco_regular': 'Regular Price (R$)', 'brand': 'Brand'}
)
fig.show()

In [ ]:
# Price by category and brand
fig = px.box(
    df[df['preco_regular'] < 500],
    x='categoria_normalizada',
    y='preco_regular',
    color='brand',
    title='Price Distribution by Category and Brand',
    color_discrete_map=BRAND_COLORS,
    labels={'categoria_normalizada': 'Category', 'preco_regular': 'Regular Price (R$)', 'brand': 'Brand'}
)
fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

In [ ]:
# Price tier distribution
tier_dist = df.groupby(['brand', 'price_tier']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
tier_dist.T.plot(kind='bar', ax=ax, color=[BRAND_COLORS['boticario'], BRAND_COLORS['natura']])
plt.title('Price Tier Distribution by Brand', fontsize=14, fontweight='bold')
plt.xlabel('Price Tier')
plt.ylabel('Number of Products')
plt.xticks(rotation=0)
plt.legend(title='Brand')

# Add percentage labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Regular vs Promotional Price scatter
df_promo = df[df['preco_promocional'].notna() & (df['preco_regular'] < 500)]

fig = px.scatter(
    df_promo,
    x='preco_regular',
    y='preco_promocional',
    color='brand',
    opacity=0.5,
    title='Regular vs Promotional Price',
    color_discrete_map=BRAND_COLORS,
    labels={'preco_regular': 'Regular Price (R$)', 'preco_promocional': 'Promotional Price (R$)'}
)

# Add diagonal line (no discount line)
fig.add_trace(go.Scatter(
    x=[0, 500], y=[0, 500],
    mode='lines',
    name='No Discount Line',
    line=dict(color='gray', dash='dash')
))

fig.show()

### 5.3 Savings Analysis

In [ ]:
# Savings statistics
savings_stats = df[df['economia'] > 0].groupby('brand')['economia'].agg(
    ['count', 'sum', 'mean', 'median', 'max']
).round(2)
savings_stats.columns = ['Products', 'Total Savings (R$)', 'Mean (R$)', 'Median (R$)', 'Max (R$)']
print("Savings Statistics by Brand:")
savings_stats

In [ ]:
# Total savings by category
category_savings = df[df['economia'] > 0].groupby('categoria_normalizada')['economia'].agg(
    ['sum', 'mean', 'count']
).sort_values('sum', ascending=False)

fig = px.bar(
    category_savings.reset_index(),
    x='categoria_normalizada',
    y='sum',
    color='mean',
    title='Total Savings Potential by Category',
    labels={'categoria_normalizada': 'Category', 'sum': 'Total Savings (R$)', 'mean': 'Avg Savings (R$)'},
    color_continuous_scale='Greens'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Savings vs Price relationship
fig = px.scatter(
    df[(df['economia'] > 0) & (df['preco_regular'] < 500)],
    x='preco_regular',
    y='economia',
    color='categoria_normalizada',
    size='desconto_calculado',
    opacity=0.6,
    title='Savings vs Regular Price by Category',
    labels={'preco_regular': 'Regular Price (R$)', 'economia': 'Savings (R$)', 
            'categoria_normalizada': 'Category', 'desconto_calculado': 'Discount %'},
    height=600
)
fig.show()

---
## 6. Category & Product Line Analysis

### 6.1 Category Distribution

In [ ]:
# Category distribution overview
category_dist = df['categoria_normalizada'].value_counts()

fig = px.pie(
    values=category_dist.values,
    names=category_dist.index,
    title='Product Distribution by Category',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Category distribution by brand
cat_brand = df.groupby(['brand', 'categoria_normalizada']).size().unstack(fill_value=0)

# Normalize to percentages
cat_brand_pct = cat_brand.div(cat_brand.sum(axis=1), axis=0) * 100

fig = px.bar(
    cat_brand_pct.T.reset_index().melt(id_vars='categoria_normalizada'),
    x='categoria_normalizada',
    y='value',
    color='brand',
    barmode='group',
    title='Category Share by Brand (%)',
    color_discrete_map=BRAND_COLORS,
    labels={'categoria_normalizada': 'Category', 'value': 'Share (%)', 'brand': 'Brand'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Category treemap
cat_summary = df.groupby(['brand', 'categoria_normalizada']).agg(
    count=('codigo', 'count'),
    avg_price=('preco_regular', 'mean')
).reset_index()

fig = px.treemap(
    cat_summary,
    path=['brand', 'categoria_normalizada'],
    values='count',
    color='avg_price',
    title='Product Portfolio Treemap',
    color_continuous_scale='RdYlGn_r',
    labels={'count': 'Products', 'avg_price': 'Avg Price (R$)'}
)
fig.show()

### 6.2 Product Line Analysis

In [ ]:
# Top 20 product lines
top_lines = df['linha'].value_counts().head(20)

fig = px.bar(
    x=top_lines.values,
    y=top_lines.index,
    orientation='h',
    title='Top 20 Product Lines by Product Count',
    labels={'x': 'Number of Products', 'y': 'Product Line'},
    color=top_lines.values,
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

In [ ]:
# Product line metrics
line_metrics = df.groupby('linha').agg(
    product_count=('codigo', 'count'),
    avg_price=('preco_regular', 'mean'),
    promo_rate=('promocao_ativa', 'mean'),
    avg_discount=('desconto_calculado', 'mean')
).round(2)

# Filter to lines with at least 10 products
line_metrics = line_metrics[line_metrics['product_count'] >= 10].sort_values('product_count', ascending=False)

# Bubble chart
fig = px.scatter(
    line_metrics.reset_index().head(30),
    x='avg_price',
    y='promo_rate',
    size='product_count',
    color='avg_discount',
    hover_name='linha',
    title='Product Lines: Price vs Promotion Rate (Top 30)',
    labels={'avg_price': 'Avg Price (R$)', 'promo_rate': 'Promotion Rate', 
            'product_count': 'Products', 'avg_discount': 'Avg Discount %'},
    color_continuous_scale='RdYlGn'
)
fig.show()

In [ ]:
# Top lines by brand
for brand in ['boticario', 'natura']:
    brand_lines = df[df['brand'] == brand]['linha'].value_counts().head(10)
    print(f"\nTop 10 Product Lines - {brand.title()}:")
    for i, (line, count) in enumerate(brand_lines.items(), 1):
        print(f"  {i}. {line}: {count} products")

### 6.3 Volume/Weight Analysis

In [ ]:
# Volume distribution by unit type
volume_dist = df[df['volume_numeric'].notna()].groupby('volume_unit')['volume_numeric'].describe()
print("Volume Statistics by Unit:")
volume_dist.round(2)

In [ ]:
# Volume distribution by category (ml only)
df_ml = df[(df['volume_unit'] == 'ml') & (df['volume_numeric'] < 1000)]

fig = px.box(
    df_ml,
    x='categoria_normalizada',
    y='volume_numeric',
    color='brand',
    title='Volume Distribution by Category (ml products < 1000ml)',
    color_discrete_map=BRAND_COLORS,
    labels={'categoria_normalizada': 'Category', 'volume_numeric': 'Volume (ml)'}
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

In [ ]:
# Price per ml analysis
df_price_per_ml = df[(df['volume_unit'] == 'ml') & (df['price_per_unit'].notna()) & (df['price_per_unit'] < 5)]

fig = px.box(
    df_price_per_ml,
    x='categoria_normalizada',
    y='price_per_unit',
    color='brand',
    title='Price per ml by Category',
    color_discrete_map=BRAND_COLORS,
    labels={'categoria_normalizada': 'Category', 'price_per_unit': 'Price per ml (R$)'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 6.4 Brand Comparison

In [ ]:
# Comprehensive brand comparison
brand_comparison = df.groupby('brand').agg(
    total_products=('codigo', 'count'),
    unique_products=('codigo', 'nunique'),
    unique_lines=('linha', 'nunique'),
    avg_price=('preco_regular', 'mean'),
    median_price=('preco_regular', 'median'),
    max_price=('preco_regular', 'max'),
    promo_rate=('promocao_ativa', 'mean'),
    avg_discount=('desconto_calculado', 'mean'),
    total_savings=('economia', 'sum'),
    avg_features=('features_count', 'mean')
).round(2).T

brand_comparison.columns = ['O Boticário', 'Natura']
print("Brand Comparison:")
brand_comparison

In [ ]:
# Radar chart for brand comparison
categories_radar = ['Avg Price', 'Promo Rate', 'Avg Discount', 'Product Lines', 'Avg Features']

# Normalize values for radar chart
boticario_values = [
    df[df['brand'] == 'boticario']['preco_regular'].mean() / 100,
    df[df['brand'] == 'boticario']['promocao_ativa'].mean(),
    df[df['brand'] == 'boticario']['desconto_calculado'].mean() / 100,
    df[df['brand'] == 'boticario']['linha'].nunique() / 100,
    df[df['brand'] == 'boticario']['features_count'].mean()
]

natura_values = [
    df[df['brand'] == 'natura']['preco_regular'].mean() / 100,
    df[df['brand'] == 'natura']['promocao_ativa'].mean(),
    df[df['brand'] == 'natura']['desconto_calculado'].mean() / 100,
    df[df['brand'] == 'natura']['linha'].nunique() / 100,
    df[df['brand'] == 'natura']['features_count'].mean()
]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=boticario_values + [boticario_values[0]],
    theta=categories_radar + [categories_radar[0]],
    fill='toself',
    name='O Boticário',
    line_color=BRAND_COLORS['boticario']
))

fig.add_trace(go.Scatterpolar(
    r=natura_values + [natura_values[0]],
    theta=categories_radar + [categories_radar[0]],
    fill='toself',
    name='Natura',
    line_color=BRAND_COLORS['natura']
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    title='Brand Comparison Radar Chart (Normalized)'
)
fig.show()

---
## 7. Promotional Strategy Analysis

### 7.1 Promotion Type Distribution

In [ ]:
# Promotion type distribution
promo_type_dist = df['promo_type'].value_counts()
print("Promotion Type Distribution:")
for ptype, count in promo_type_dist.items():
    pct = count / len(df) * 100
    print(f"  {ptype}: {count:,} ({pct:.1f}%)")

In [ ]:
# Promotion type by brand
promo_by_brand = df.groupby(['brand', 'promo_type']).size().unstack(fill_value=0)

fig = px.bar(
    promo_by_brand.T.reset_index().melt(id_vars='promo_type'),
    x='promo_type',
    y='value',
    color='brand',
    barmode='group',
    title='Promotion Type Distribution by Brand',
    color_discrete_map=BRAND_COLORS,
    labels={'promo_type': 'Promotion Type', 'value': 'Number of Products', 'brand': 'Brand'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Heatmap: Category x Promotion Type
cat_promo = pd.crosstab(df['categoria_normalizada'], df['promo_type'], normalize='index') * 100

plt.figure(figsize=(12, 8))
sns.heatmap(cat_promo, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('Promotion Type by Category (% of category)', fontsize=14, fontweight='bold')
plt.xlabel('Promotion Type')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

### 7.2 Progressive Discount Analysis

In [ ]:
# Filter progressive discount products
df_progressive = df[df['promo_type'] == 'progressive_discount'].copy()
print(f"Products with Progressive Discount: {len(df_progressive)}")

# Analyze discount tiers
def analyze_tiers(tiers):
    if not isinstance(tiers, dict) or len(tiers) == 0:
        return None, None, None
    values = list(tiers.values())
    return len(tiers), min(values) if values else None, max(values) if values else None

df_progressive[['tier_count', 'min_tier', 'max_tier']] = df_progressive['discount_tiers'].apply(
    lambda x: pd.Series(analyze_tiers(x))
)

print("\nTier Statistics:")
print(df_progressive[['tier_count', 'min_tier', 'max_tier']].describe())

In [ ]:
# Tier configuration distribution
if len(df_progressive) > 0:
    tier_configs = df_progressive['tier_count'].value_counts().sort_index()
    
    fig = px.bar(
        x=tier_configs.index.astype(str),
        y=tier_configs.values,
        title='Progressive Discount: Number of Tiers Distribution',
        labels={'x': 'Number of Tiers', 'y': 'Product Count'},
        color=tier_configs.values,
        color_continuous_scale='Purples'
    )
    fig.show()

### 7.3 Combo Analysis

In [ ]:
# Filter combo products
df_combo = df[df['promo_type'] == 'combo'].copy()
print(f"Products with Combo Promotions: {len(df_combo)}")

# Analyze combo relationships
combo_relationships = []
for _, row in df_combo.iterrows():
    codes = row.get('combo_codes', [])
    if isinstance(codes, list) and len(codes) > 0:
        for code in codes:
            combo_relationships.append({
                'source': row['codigo'],
                'target': code,
                'brand': row['brand'],
                'category': row['categoria_normalizada']
            })

df_combos = pd.DataFrame(combo_relationships)
print(f"\nCombo Relationships Found: {len(df_combos)}")

In [ ]:
# Combo network visualization (if relationships exist)
if len(df_combos) > 0:
    # Create network graph
    G = nx.from_pandas_edgelist(df_combos.head(100), 'source', 'target')
    
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=2, iterations=50)
    nx.draw(G, pos, 
            node_color='lightblue', 
            node_size=100,
            edge_color='gray',
            alpha=0.7,
            with_labels=True,
            font_size=8)
    plt.title('Product Combo Network (First 100 relationships)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No combo relationships found in the data.")

### 7.4 Promotion Trends Over Cycles

In [ ]:
# Promotion rate by cycle
cycle_promo = df.groupby(['brand', 'cycle']).agg(
    total_products=('codigo', 'count'),
    products_with_promo=('promocao_ativa', 'sum'),
    avg_discount=('desconto_calculado', 'mean')
).reset_index()

cycle_promo['promo_rate'] = cycle_promo['products_with_promo'] / cycle_promo['total_products'] * 100

fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Promotion Rate by Cycle', 'Average Discount by Cycle'),
                    vertical_spacing=0.15)

for brand in ['boticario', 'natura']:
    brand_data = cycle_promo[cycle_promo['brand'] == brand]
    
    fig.add_trace(
        go.Scatter(x=brand_data['cycle'], y=brand_data['promo_rate'],
                   mode='lines+markers', name=f'{brand.title()} - Promo Rate',
                   line=dict(color=BRAND_COLORS[brand])),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=brand_data['cycle'], y=brand_data['avg_discount'],
                   mode='lines+markers', name=f'{brand.title()} - Avg Discount',
                   line=dict(color=BRAND_COLORS[brand], dash='dash')),
        row=2, col=1
    )

fig.update_layout(height=600, title_text='Promotional Trends Over Sales Cycles')
fig.update_xaxes(title_text='Cycle', row=2, col=1)
fig.update_yaxes(title_text='Promotion Rate (%)', row=1, col=1)
fig.update_yaxes(title_text='Avg Discount (%)', row=2, col=1)
fig.show()

In [ ]:
# Category promotion intensity heatmap over cycles
cat_cycle_promo = df.groupby(['categoria_normalizada', 'cycle'])['promocao_ativa'].mean().unstack(fill_value=0) * 100

plt.figure(figsize=(14, 8))
sns.heatmap(cat_cycle_promo, cmap='YlOrRd', annot=True, fmt='.0f', linewidths=0.5)
plt.title('Category Promotion Rate by Cycle (%)', fontsize=14, fontweight='bold')
plt.xlabel('Cycle')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

---
## 8. Data Quality Analysis

### 8.1 Validation Alerts Analysis

In [ ]:
# Alert statistics
alert_stats = {
    'Total Products': len(df),
    'Products with Alerts': (df['alert_count'] > 0).sum(),
    'Products with Errors': df['has_errors'].sum(),
    'Products with Warnings': df['has_warnings'].sum(),
    'Alert Rate (%)': round((df['alert_count'] > 0).mean() * 100, 2),
    'Error Rate (%)': round(df['has_errors'].mean() * 100, 2)
}

print("Data Quality Overview:")
for key, value in alert_stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Extract individual alerts for analysis
all_alerts = []
for _, row in df.iterrows():
    alerts = row.get('alertas', []) or []
    for alert in alerts:
        all_alerts.append({
            'product_code': row['codigo'],
            'brand': row['brand'],
            'category': row['categoria_normalizada'],
            'level': alert.get('level'),
            'field': alert.get('field'),
            'message': alert.get('message')
        })

df_alerts = pd.DataFrame(all_alerts)
print(f"Total alerts extracted: {len(df_alerts)}")

In [ ]:
# Alert distribution by level and field
if len(df_alerts) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # By level
    level_counts = df_alerts['level'].value_counts()
    colors_level = {'error': 'red', 'warning': 'orange', 'info': 'blue'}
    axes[0].pie(level_counts, labels=level_counts.index, autopct='%1.1f%%',
                colors=[colors_level.get(l, 'gray') for l in level_counts.index])
    axes[0].set_title('Alerts by Level', fontweight='bold')
    
    # By field
    field_counts = df_alerts['field'].value_counts().head(10)
    axes[1].barh(field_counts.index, field_counts.values, color='steelblue')
    axes[1].set_title('Top 10 Fields with Alerts', fontweight='bold')
    axes[1].set_xlabel('Number of Alerts')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
else:
    print("No alerts found in the dataset.")

### 8.2 Missing Data Analysis

In [ ]:
# Missing data heatmap
key_fields = ['codigo', 'nome', 'linha', 'categoria_normalizada', 'volume_peso',
              'preco_regular', 'preco_promocional', 'desconto_percentual', 'economia']

missing_matrix = df[key_fields].isnull()

plt.figure(figsize=(12, 6))
sns.heatmap(missing_matrix.head(500).T, cmap='YlOrRd', cbar_kws={'label': 'Missing'})
plt.title('Missing Data Pattern (First 500 Products)', fontsize=14, fontweight='bold')
plt.xlabel('Product Index')
plt.ylabel('Field')
plt.tight_layout()
plt.show()

In [ ]:
# Missing data by brand
missing_by_brand = df.groupby('brand')[key_fields].apply(lambda x: x.isnull().mean() * 100).round(2)

fig = px.bar(
    missing_by_brand.T.reset_index().melt(id_vars='index'),
    x='index',
    y='value',
    color='brand',
    barmode='group',
    title='Missing Data Rate by Field and Brand (%)',
    color_discrete_map=BRAND_COLORS,
    labels={'index': 'Field', 'value': 'Missing Rate (%)', 'brand': 'Brand'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 8.3 Category Normalization Quality

In [ ]:
# Raw vs Normalized category mapping
category_mapping = df.groupby(['categoria', 'categoria_normalizada']).size().reset_index(name='count')
category_mapping = category_mapping.sort_values('count', ascending=False)

print(f"Unique Raw Categories: {df['categoria'].nunique()}")
print(f"Unique Normalized Categories: {df['categoria_normalizada'].nunique()}")
print(f"\nTop 20 Category Mappings:")
category_mapping.head(20)

In [ ]:
# Check for "Outros" category (unmapped)
outros_count = (df['categoria_normalizada'] == 'Outros').sum()
outros_pct = outros_count / len(df) * 100

print(f"Products in 'Outros' category: {outros_count} ({outros_pct:.2f}%)")

if outros_count > 0:
    print("\nRaw categories mapped to 'Outros':")
    outros_raw = df[df['categoria_normalizada'] == 'Outros']['categoria'].value_counts().head(10)
    for cat, count in outros_raw.items():
        print(f"  {cat}: {count}")

---
## 9. Advanced Analytics

### 9.1 Price Clustering

In [ ]:
# Prepare data for clustering
cluster_features = ['preco_regular', 'desconto_calculado', 'volume_numeric']
df_cluster = df[cluster_features].dropna().copy()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print(f"Products for clustering: {len(df_cluster)}")

In [ ]:
# Find optimal number of clusters (Elbow Method)
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontweight='bold')

# Silhouette plot
axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Optimal K (by silhouette): {K_range[np.argmax(silhouettes)]}")

In [ ]:
# Apply K-Means with optimal K
optimal_k = 4  # Adjust based on elbow/silhouette analysis

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

# Cluster profiles
cluster_profiles = df_cluster.groupby('cluster').agg(
    count=('preco_regular', 'count'),
    avg_price=('preco_regular', 'mean'),
    avg_discount=('desconto_calculado', 'mean'),
    avg_volume=('volume_numeric', 'mean')
).round(2)

# Name clusters
cluster_names = {}
for idx, row in cluster_profiles.iterrows():
    if row['avg_price'] < 50:
        cluster_names[idx] = 'Budget'
    elif row['avg_price'] < 100:
        cluster_names[idx] = 'Economy'
    elif row['avg_price'] < 200:
        cluster_names[idx] = 'Premium'
    else:
        cluster_names[idx] = 'Luxury'

cluster_profiles['cluster_name'] = cluster_profiles.index.map(cluster_names)
print("Cluster Profiles:")
cluster_profiles

In [ ]:
# Visualize clusters
df_cluster['cluster_name'] = df_cluster['cluster'].map(cluster_names)

fig = px.scatter(
    df_cluster,
    x='preco_regular',
    y='desconto_calculado',
    color='cluster_name',
    size='volume_numeric',
    opacity=0.6,
    title='Product Clusters: Price vs Discount',
    labels={'preco_regular': 'Regular Price (R$)', 'desconto_calculado': 'Discount (%)',
            'cluster_name': 'Cluster', 'volume_numeric': 'Volume'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

### 9.2 NLP Analysis on Product Names

In [ ]:
# Text preprocessing
import string

def preprocess_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['nome_clean'] = df['nome'].apply(preprocess_text)

# Word frequency
all_words = ' '.join(df['nome_clean']).split()
word_freq = Counter(all_words)

# Remove common stopwords
stopwords_pt = {'de', 'da', 'do', 'das', 'dos', 'para', 'com', 'em', 'e', 'o', 'a', 'os', 'as', 
                'um', 'uma', 'ml', 'g', 'gr', 'x', 'kit', 'set'}
word_freq_filtered = {w: c for w, c in word_freq.items() if w not in stopwords_pt and len(w) > 2}

top_words = dict(sorted(word_freq_filtered.items(), key=lambda x: x[1], reverse=True)[:30])
print("Top 30 Words in Product Names:")
for word, count in top_words.items():
    print(f"  {word}: {count}")

In [ ]:
# Word cloud for all products
wordcloud = WordCloud(
    width=1200, height=600,
    background_color='white',
    colormap='viridis',
    max_words=100
).generate_from_frequencies(word_freq_filtered)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Product Names Word Cloud', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Word clouds by brand
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, brand in enumerate(['boticario', 'natura']):
    brand_words = ' '.join(df[df['brand'] == brand]['nome_clean']).split()
    brand_freq = Counter(brand_words)
    brand_freq_filtered = {w: c for w, c in brand_freq.items() if w not in stopwords_pt and len(w) > 2}
    
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap='Greens' if brand == 'boticario' else 'Oranges',
        max_words=50
    ).generate_from_frequencies(brand_freq_filtered)
    
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].axis('off')
    axes[idx].set_title(f'{brand.title()} - Top Words', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Extract key attributes from product names
# Scent families
scent_keywords = {
    'floral': ['rosa', 'flor', 'jasmin', 'orquídea', 'gardênia', 'violeta', 'lírio'],
    'frutal': ['frutas', 'morango', 'cereja', 'maçã', 'pêssego', 'framboesa', 'amora', 'uva'],
    'amadeirado': ['madeira', 'sândalo', 'cedro', 'canela', 'âmbar'],
    'cítrico': ['limão', 'laranja', 'bergamota', 'tangerina', 'citrus'],
    'gourmand': ['chocolate', 'baunilha', 'caramelo', 'café', 'mel', 'doce'],
    'fresco': ['menta', 'água', 'marine', 'fresh', 'acqua']
}

def detect_scent_family(text):
    text = str(text).lower()
    for family, keywords in scent_keywords.items():
        if any(kw in text for kw in keywords):
            return family
    return 'other'

df['scent_family'] = df['nome'].apply(detect_scent_family)

scent_dist = df['scent_family'].value_counts()
print("Scent Family Distribution:")
for family, count in scent_dist.items():
    pct = count / len(df) * 100
    print(f"  {family}: {count} ({pct:.1f}%)")

In [ ]:
# Scent family by category
scent_category = pd.crosstab(df['categoria_normalizada'], df['scent_family'], normalize='index') * 100

fig = px.imshow(
    scent_category,
    labels=dict(x='Scent Family', y='Category', color='Percentage'),
    title='Scent Family Distribution by Category (%)',
    color_continuous_scale='Blues'
)
fig.update_layout(height=500)
fig.show()

### 9.3 Product Similarity & Recommendations

In [ ]:
# TF-IDF on product names
tfidf = TfidfVectorizer(max_features=500, stop_words=list(stopwords_pt))
tfidf_matrix = tfidf.fit_transform(df['nome_clean'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

In [ ]:
# Calculate cosine similarity (sample for memory efficiency)
sample_size = 1000
sample_indices = np.random.choice(len(df), min(sample_size, len(df)), replace=False)
sample_matrix = tfidf_matrix[sample_indices]

similarity_matrix = cosine_similarity(sample_matrix)
print(f"Similarity Matrix Shape: {similarity_matrix.shape}")

In [ ]:
# Recommendation function
def get_similar_products(product_idx, n=5):
    """
    Get N most similar products to a given product.
    """
    # Find position in sample
    if product_idx not in sample_indices:
        return "Product not in sample"
    
    sample_pos = np.where(sample_indices == product_idx)[0][0]
    similarities = similarity_matrix[sample_pos]
    
    # Get top N similar (excluding self)
    similar_indices = np.argsort(similarities)[::-1][1:n+1]
    
    results = []
    for idx in similar_indices:
        original_idx = sample_indices[idx]
        results.append({
            'nome': df.iloc[original_idx]['nome'],
            'categoria': df.iloc[original_idx]['categoria_normalizada'],
            'preco': df.iloc[original_idx]['preco_regular'],
            'similarity': similarities[idx]
        })
    
    return pd.DataFrame(results)

# Example: Get similar products for a random product
random_product = sample_indices[42]
print(f"Product: {df.iloc[random_product]['nome']}")
print(f"Category: {df.iloc[random_product]['categoria_normalizada']}")
print(f"Price: {format_currency(df.iloc[random_product]['preco_regular'])}")
print("\nSimilar Products:")
get_similar_products(random_product, n=5)

In [ ]:
# t-SNE visualization of product embeddings
print("Computing t-SNE (this may take a moment)...")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_result = tsne.fit_transform(sample_matrix.toarray())

df_tsne = pd.DataFrame({
    'x': tsne_result[:, 0],
    'y': tsne_result[:, 1],
    'category': df.iloc[sample_indices]['categoria_normalizada'].values,
    'brand': df.iloc[sample_indices]['brand'].values,
    'nome': df.iloc[sample_indices]['nome'].values
})

fig = px.scatter(
    df_tsne,
    x='x', y='y',
    color='category',
    hover_data=['nome', 'brand'],
    title='Product Embeddings (t-SNE)',
    opacity=0.7,
    height=700
)
fig.show()

### 9.4 Market Basket Analysis

In [ ]:
# Create page co-occurrence matrix (products on same page)
page_products = df.groupby(['source_file', 'pagina'])['codigo'].apply(list).reset_index()
page_products = page_products[page_products['codigo'].apply(len) > 1]

print(f"Pages with multiple products: {len(page_products)}")

In [ ]:
# Calculate category co-occurrence
category_pairs = []

for _, row in page_products.iterrows():
    products = row['codigo']
    categories = df[df['codigo'].isin(products)]['categoria_normalizada'].unique()
    
    if len(categories) > 1:
        for i in range(len(categories)):
            for j in range(i+1, len(categories)):
                category_pairs.append((categories[i], categories[j]))

pair_counts = Counter(category_pairs)
print("Top 15 Category Co-occurrences (on same page):")
for pair, count in pair_counts.most_common(15):
    print(f"  {pair[0]} + {pair[1]}: {count}")

In [ ]:
# Category co-occurrence heatmap
categories_unique = df['categoria_normalizada'].unique()
cooccurrence_matrix = pd.DataFrame(0, index=categories_unique, columns=categories_unique)

for (cat1, cat2), count in pair_counts.items():
    cooccurrence_matrix.loc[cat1, cat2] = count
    cooccurrence_matrix.loc[cat2, cat1] = count

plt.figure(figsize=(12, 10))
mask = np.zeros_like(cooccurrence_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(cooccurrence_matrix, mask=mask, annot=True, fmt='d', cmap='YlGnBu', 
            linewidths=0.5, square=True)
plt.title('Category Co-occurrence Matrix (Products on Same Page)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Conclusions & Insights

In [ ]:
# Summary statistics
print("=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print("\n📊 DATASET OVERVIEW")
print(f"   • Total Products Analyzed: {len(df):,}")
print(f"   • Brands: O Boticário & Natura")
print(f"   • Categories: {df['categoria_normalizada'].nunique()}")
print(f"   • Product Lines: {df['linha'].nunique()}")

print("\n💰 PRICING INSIGHTS")
print(f"   • Average Price: {format_currency(df['preco_regular'].mean())}")
print(f"   • Median Price: {format_currency(df['preco_regular'].median())}")
print(f"   • Price Range: {format_currency(df['preco_regular'].min())} - {format_currency(df['preco_regular'].max())}")

print("\n🏷️ PROMOTIONAL INSIGHTS")
print(f"   • Products with Promotions: {df['promocao_ativa'].sum():,} ({df['promocao_ativa'].mean()*100:.1f}%)")
print(f"   • Average Discount: {df[df['desconto_calculado'] > 0]['desconto_calculado'].mean():.1f}%")
print(f"   • Total Potential Savings: {format_currency(df['economia'].sum())}")

print("\n📈 TOP CATEGORIES")
top_cats = df['categoria_normalizada'].value_counts().head(5)
for cat, count in top_cats.items():
    pct = count / len(df) * 100
    print(f"   • {cat}: {count:,} products ({pct:.1f}%)")

print("\n✅ DATA QUALITY")
print(f"   • Products with Alerts: {(df['alert_count'] > 0).sum():,}")
print(f"   • Error Rate: {df['has_errors'].mean()*100:.2f}%")
print(f"   • Completeness: {(1 - df[['codigo', 'nome', 'preco_regular']].isnull().any(axis=1).mean())*100:.1f}%")

In [ ]:
print("\n" + "=" * 70)
print("BUSINESS RECOMMENDATIONS")
print("=" * 70)

print("""
1. PRICING OPTIMIZATION
   • Analyze price-per-unit metrics across categories to identify
     premium pricing opportunities
   • Consider tiered pricing strategies based on cluster analysis

2. PROMOTIONAL STRATEGY
   • High-performing categories for promotions identified
   • Progressive discounts show potential for volume increases
   • Combo opportunities based on page co-occurrence patterns

3. PORTFOLIO MANAGEMENT
   • Top product lines identified for focus and investment
   • Category gaps between brands present cross-learning opportunities
   • Scent family analysis supports product development decisions

4. DATA QUALITY IMPROVEMENTS
   • Address validation alerts to improve extraction accuracy
   • Reduce "Outros" category through better normalization rules
   • Standardize volume/weight capture for better unit analysis
""")

In [ ]:
print("\n" + "=" * 70)
print("FUTURE ANALYSIS OPPORTUNITIES")
print("=" * 70)

print("""
• Price elasticity modeling with external sales data
• Seasonal pattern analysis across multiple years
• Competitive positioning analysis (if competitor data available)
• Customer segmentation with purchase data integration
• Deep learning for image-based product classification
• Natural language generation for product descriptions
""")

print("\n" + "=" * 70)
print("Analysis Complete!")
print("=" * 70)